# 01 — Train, Validation, and Test Split

This notebook splits the joined click data into 70% training, 15% validation, and 15% test data. The prediction target is `clicked`.

A chronological split is used because the model will predict future clicks from past data.

## Install required packages

In [ ]:
%pip install pandas matplotlib

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

current_dir = Path.cwd().resolve()
project_root = next(
    (path for path in [current_dir, *current_dir.parents] if (path / "data" / "interim").exists()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Could not find data/interim. Start Jupyter inside this project.")

input_path = project_root / "data" / "interim" / "click_data_joined.csv"
processed_dir = project_root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)
print(f"Input: {input_path}")
print(f"Output directory: {processed_dir}")

## Load and validate the modeling data

In [ ]:
df = pd.read_csv(input_path, parse_dates=["event_ts"])
target = "clicked"

required_columns = {"impression_id", "event_ts", target}
missing_required = required_columns.difference(df.columns)
if missing_required:
    raise ValueError(f"Missing required columns: {sorted(missing_required)}")
if df[target].isna().any():
    raise ValueError("The clicked target contains missing values.")
if not set(df[target].unique()).issubset({0, 1}):
    raise ValueError("The clicked target must contain only 0 and 1.")
if df["event_ts"].isna().any():
    raise ValueError("event_ts contains missing or invalid timestamps.")
if df["impression_id"].duplicated().any():
    raise ValueError("impression_id must be unique before splitting.")

print(f"Rows available: {len(df):,}")
print(f"Columns available: {df.shape[1]}")
print(f"Overall click rate: {df[target].mean():.2%}")

## Create the chronological split

Rows are sorted from oldest to newest. The earliest 70% become training data, the next 15% become validation data, and the newest 15% become test data.

In [ ]:
df = df.sort_values(["event_ts", "impression_id"]).reset_index(drop=True)

n_rows = len(df)
train_end = int(n_rows * 0.70)
validation_end = train_end + int(n_rows * 0.15)

train_df = df.iloc[:train_end].copy()
validation_df = df.iloc[train_end:validation_end].copy()
test_df = df.iloc[validation_end:].copy()

assert len(train_df) + len(validation_df) + len(test_df) == n_rows
assert set(train_df["impression_id"]).isdisjoint(validation_df["impression_id"])
assert set(train_df["impression_id"]).isdisjoint(test_df["impression_id"])
assert set(validation_df["impression_id"]).isdisjoint(test_df["impression_id"])

## Review split sizes, dates, and target rates

In [ ]:
splits = {"train": train_df, "validation": validation_df, "test": test_df}
split_summary = pd.DataFrame([
    {
        "split": name,
        "rows": len(split_df),
        "share": len(split_df) / n_rows,
        "start_time": split_df["event_ts"].min(),
        "end_time": split_df["event_ts"].max(),
        "non_clicks": int((split_df[target] == 0).sum()),
        "clicks": int((split_df[target] == 1).sum()),
        "click_rate": split_df[target].mean(),
    }
    for name, split_df in splits.items()
]).set_index("split")
display(split_summary)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
split_summary["rows"].plot(kind="bar", ax=axes[0], color=["#4C78A8", "#F58518", "#54A24B"])
axes[0].set(title="Rows in each split", xlabel="Split", ylabel="Rows")
split_summary["click_rate"].plot(kind="bar", ax=axes[1], color=["#4C78A8", "#F58518", "#54A24B"])
axes[1].axhline(df[target].mean(), color="#E45756", linestyle="--", label="Overall CTR")
axes[1].set(title="Click rate in each split", xlabel="Split", ylabel="Click rate")
axes[1].legend()
plt.tight_layout()
plt.show()

## Separate predictors and target

The full split files are saved below, but these `X` and `y` objects show exactly what will be passed to a modeling pipeline. `impression_id` identifies rows and is excluded from predictors.

In [ ]:
excluded_from_features = ["impression_id", target]
feature_columns = [column for column in df.columns if column not in excluded_from_features]

X_train = train_df[feature_columns].copy()
y_train = train_df[target].copy()
X_validation = validation_df[feature_columns].copy()
y_validation = validation_df[target].copy()
X_test = test_df[feature_columns].copy()
y_test = test_df[target].copy()

pd.DataFrame({
    "object": ["X_train", "y_train", "X_validation", "y_validation", "X_test", "y_test"],
    "shape": [X_train.shape, y_train.shape, X_validation.shape, y_validation.shape, X_test.shape, y_test.shape],
})

## Save the split datasets

In [ ]:
output_paths = {
    "train": processed_dir / "train.csv",
    "validation": processed_dir / "validation.csv",
    "test": processed_dir / "test.csv",
}

for name, split_df in splits.items():
    split_df.to_csv(output_paths[name], index=False)
    print(f"Saved {name}: {len(split_df):,} rows -> {output_paths[name]}")

## Key findings for preprocessing

In [ ]:
same_boundary_timestamp = (
    train_df["event_ts"].max() == validation_df["event_ts"].min()
    or validation_df["event_ts"].max() == test_df["event_ts"].min()
)
largest_ctr_change = (split_summary["click_rate"] - df[target].mean()).abs().max()

display(Markdown(f"""
### Split recap

- **Training:** {len(train_df):,} rows ({len(train_df) / n_rows:.1%}) from {train_df['event_ts'].min()} through {train_df['event_ts'].max()}; CTR is {train_df[target].mean():.2%}.
- **Validation:** {len(validation_df):,} rows ({len(validation_df) / n_rows:.1%}) from {validation_df['event_ts'].min()} through {validation_df['event_ts'].max()}; CTR is {validation_df[target].mean():.2%}.
- **Test:** {len(test_df):,} rows ({len(test_df) / n_rows:.1%}) from {test_df['event_ts'].min()} through {test_df['event_ts'].max()}; CTR is {test_df[target].mean():.2%}.
- `clicked` is the target. `impression_id` is kept for tracking but removed from model features.
- The largest split CTR difference from the overall CTR is {largest_ctr_change:.2%}. Large differences may indicate change over time.
- Same timestamp at a split boundary: **{'yes' if same_boundary_timestamp else 'no'}**. If yes, rows are still separated deterministically by `impression_id`.
- All three files are saved in `data/processed`. Preprocessing rules must be fitted on training data only, then applied unchanged to validation and test data.
- Use validation data for model and threshold choices. Keep test data untouched until the final evaluation.
"""))